# Prophet Forecasting for All Electricity Clients

This notebook trains a Prophet baseline model for every client column in `dataset/LD2011_2014.txt`.

The deeper EDA shows three important modelling facts:

- the dataset has 370 client load series at 15-minute frequency,
- many clients are late starters, so leading zeros should not be treated as normal consumption,
- the series contain strong sub-daily, daily, weekly, monthly, and yearly seasonal signals.


## 1. Setup

In [ ]:
from pathlib import Path
import os
import time
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

In [ ]:
# Paths
DATA_PATH = Path("dataset/LD2011_2014.txt")
OUTPUT_DIR = Path("outputs/prophet_all")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Data frequencies
RAW_FREQ = "15min"
FORECAST_FREQ = "h"       # Hourly aggregation keeps the main seasonalities and makes 370 Prophet fits practical.
RESAMPLE_METHOD = "mean"

# Forecasting setup
TEST_DAYS = 30
FUTURE_DAYS = 7
MIN_HISTORY_DAYS = 90      # Shorter histories are marked as skipped instead of forcing an unreliable baseline.

# Preprocessing setup based on EDA findings
REPLACE_NON_POSITIVE_WITH_NA = True
INTERPOLATE_MISSING = True
CLIP_EXTREME_OUTLIERS = True
LOWER_CLIP_Q = 0.001
UPPER_CLIP_Q = 0.999

# Full run control
# Default is 0, which means use all client columns. Set to a small number for a quick smoke test.
SMOKE_TEST_N = int(os.getenv("PROPHET_ALL_SMOKE_N", "0"))

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

## 2. Load and Validate the Full Dataset

In [ ]:
raw_df = pd.read_csv(
    DATA_PATH,
    delimiter=";",
    decimal=",",
    parse_dates=[0],
    index_col=0,
    low_memory=False,
)

raw_df.index.name = "ds"
raw_df = raw_df.sort_index()
raw_df = raw_df[~raw_df.index.duplicated(keep="last")]
raw_df = raw_df.apply(pd.to_numeric, errors="coerce")

target_columns = list(raw_df.columns)

print(f"Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} client columns")
print(f"Time range: {raw_df.index.min()} to {raw_df.index.max()}")
print(f"Missing values: {raw_df.isna().sum().sum():,}")
print(f"Most common time step: {pd.Series(raw_df.index).diff().dropna().mode().iloc[0]}")
print(f"Non-{RAW_FREQ} gaps: {(pd.Series(raw_df.index).diff().dropna() != pd.Timedelta(RAW_FREQ)).sum():,}")
raw_df.head()

In [ ]:
selected_columns = target_columns if SMOKE_TEST_N <= 0 else target_columns[:SMOKE_TEST_N]

print(f"Total available client columns: {len(target_columns)}")
print(f"Columns selected for this run: {len(selected_columns)}")
if SMOKE_TEST_N > 0:
    print(f"Smoke-test mode is active: first {SMOKE_TEST_N} columns only")

## 3. Client Activity Summary

The deep EDA showed that many clients start later in the dataset. For those clients, the long leading zero stretches are treated as inactive history, not as true zero consumption.


In [ ]:
activity_rows = []
for col in target_columns:
    positive_mask = raw_df[col] > 0
    first_active = raw_df.index[positive_mask.argmax()] if positive_mask.any() else pd.NaT
    last_active = raw_df.index[len(positive_mask) - positive_mask[::-1].argmax() - 1] if positive_mask.any() else pd.NaT
    activity_rows.append(
        {
            "client": col,
            "first_active": first_active,
            "last_active": last_active,
            "positive_readings": int(positive_mask.sum()),
            "zero_fraction": float((raw_df[col] == 0).mean()),
            "mean_load": float(raw_df[col].mean()),
            "total_load": float(raw_df[col].sum()),
        }
    )

activity_df = pd.DataFrame(activity_rows).set_index("client")
activity_df["active_from_day_1"] = activity_df["first_active"] <= raw_df.index.min() + pd.Timedelta(days=1)
activity_df["history_days_after_first_active"] = (
    raw_df.index.max() - activity_df["first_active"]
).dt.total_seconds() / (24 * 3600)

print(f"Always active clients: {activity_df['active_from_day_1'].sum()}")
print(f"Late starters: {(~activity_df['active_from_day_1']).sum()}")
print(f"Never active clients: {activity_df['first_active'].isna().sum()}")
activity_df.sort_values("total_load", ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

activity_df["zero_fraction"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Zero Fraction Across Clients")
axes[0].set_xlabel("Fraction of readings equal to zero")
axes[0].set_ylabel("Number of clients")

activity_df["history_days_after_first_active"].hist(bins=40, ax=axes[1], color="coral", edgecolor="black")
axes[1].axvline(MIN_HISTORY_DAYS, color="black", linestyle="--", label=f"{MIN_HISTORY_DAYS} days")
axes[1].set_title("Available Active History")
axes[1].set_xlabel("Days after first positive reading")
axes[1].set_ylabel("Number of clients")
axes[1].legend()

plt.tight_layout()

## 4. Activation-Aware Preprocessing

Preprocessing decisions:

- enforce a regular 15-minute grid,
- replace non-positive readings with missing values because the EDA found many activation-related zero stretches and a few zero DST/outage artifacts,
- resample each client to hourly mean load,
- interpolate missing values only after each client's first active timestamp,
- optionally clip extreme hourly outliers per client using very conservative quantiles.

The resulting hourly table can contain `NaN` values before each late-starting client's activation date. Those rows are dropped per client inside the modelling loop.


In [ ]:
def preprocess_all_clients(raw: pd.DataFrame, activity: pd.DataFrame) -> pd.DataFrame:
    regular = raw.asfreq(RAW_FREQ)
    cleaned = regular.copy()

    if REPLACE_NON_POSITIVE_WITH_NA:
        cleaned = cleaned.mask(cleaned <= 0)

    if RESAMPLE_METHOD == "mean":
        hourly = cleaned.resample(FORECAST_FREQ).mean()
    elif RESAMPLE_METHOD == "sum":
        hourly = cleaned.resample(FORECAST_FREQ).sum(min_count=1)
    else:
        raise ValueError("RESAMPLE_METHOD must be either 'mean' or 'sum'.")

    hourly.index.name = "ds"
    processed = hourly.copy()

    for col in processed.columns:
        first_active = activity.loc[col, "first_active"]
        if pd.isna(first_active):
            processed[col] = np.nan
            continue

        active_start = first_active.floor(FORECAST_FREQ)
        s = processed[col].copy()
        s.loc[s.index < active_start] = np.nan

        active_part = s.loc[s.index >= active_start].copy()
        if INTERPOLATE_MISSING:
            active_part = active_part.interpolate(method="time").ffill().bfill()

        if CLIP_EXTREME_OUTLIERS and active_part.notna().sum() > 10:
            lower, upper = active_part.quantile([LOWER_CLIP_Q, UPPER_CLIP_Q])
            active_part = active_part.clip(lower=lower, upper=upper)

        s.loc[s.index >= active_start] = active_part
        processed[col] = s

    return processed

processed_hourly = preprocess_all_clients(raw_df, activity_df)

print(f"Processed hourly shape: {processed_hourly.shape}")
print(f"Date range: {processed_hourly.index.min()} to {processed_hourly.index.max()}")
print(f"Total missing hourly cells: {processed_hourly.isna().sum().sum():,}")
processed_hourly.head()

In [ ]:
preprocess_summary = activity_df.copy()
preprocess_summary["hourly_non_missing"] = processed_hourly.notna().sum()
preprocess_summary["hourly_missing"] = processed_hourly.isna().sum()
preprocess_summary["hourly_start"] = processed_hourly.apply(lambda s: s.first_valid_index())
preprocess_summary["hourly_end"] = processed_hourly.apply(lambda s: s.last_valid_index())
preprocess_summary["hourly_history_days"] = (
    preprocess_summary["hourly_end"] - preprocess_summary["hourly_start"]
).dt.total_seconds() / (24 * 3600)
preprocess_summary["eligible_for_prophet"] = preprocess_summary["hourly_history_days"] >= MIN_HISTORY_DAYS

print(preprocess_summary["eligible_for_prophet"].value_counts().rename(index={True: "eligible", False: "not eligible"}))
preprocess_summary.sort_values("total_load", ascending=False).head(10)

In [ ]:
top_clients = preprocess_summary.sort_values("total_load", ascending=False).head(10).index.tolist()
plot_cols = [col for col in top_clients if col in selected_columns][:5]

fig, ax = plt.subplots(figsize=(14, 5))
for col in plot_cols:
    ax.plot(processed_hourly.index, processed_hourly[col], linewidth=0.6, label=col)
ax.set_title("Preprocessed Hourly Load for High-Load Clients")
ax.set_xlabel("Date")
ax.set_ylabel("Hourly mean load")
ax.legend(ncol=2)
plt.tight_layout()

## 5. Prophet Modelling Functions

One independent Prophet model is fitted for each client.

The model includes:

- sub-daily seasonalities inspired by the spectral EDA: 6-hour, 8-hour, and 12-hour patterns,
- built-in daily, weekly, and yearly seasonalities,
- an additional monthly seasonality.


In [ ]:
try:
    from prophet import Prophet
except ImportError as exc:
    raise ImportError(
        "Prophet is required for this notebook. Install it with `%pip install prophet` and rerun."
    ) from exc

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda iterable, **kwargs: iterable

In [ ]:
def build_prophet_model() -> Prophet:
    model = Prophet(
        growth="linear",
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode="additive",
        seasonality_prior_scale=10.0,
        changepoint_prior_scale=0.05,
        interval_width=0.80,
    )
    model.add_seasonality(name="six_hour", period=0.25, fourier_order=3)
    model.add_seasonality(name="eight_hour", period=1 / 3, fourier_order=3)
    model.add_seasonality(name="semi_daily", period=0.5, fourier_order=3)
    model.add_seasonality(name="monthly", period=30.5, fourier_order=5)
    return model


def prepare_client_frame(processed: pd.DataFrame, client: str) -> pd.DataFrame:
    client_df = processed[[client]].dropna().rename(columns={client: "y"}).reset_index()
    client_df["y"] = client_df["y"].astype(float)
    return client_df


def evaluate_predictions(actual: pd.Series, predicted: pd.Series) -> dict:
    actual = actual.astype(float)
    predicted = predicted.astype(float).clip(lower=0)
    absolute_error = np.abs(actual - predicted)
    denominator = np.abs(actual).sum()
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "mape_pct": mean_absolute_percentage_error(actual, predicted) * 100,
        "wape_pct": np.nan if denominator == 0 else absolute_error.sum() / denominator * 100,
    }


def fit_evaluate_forecast_client(client: str, processed: pd.DataFrame) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    client_df = prepare_client_frame(processed, client)
    history_days = (client_df["ds"].max() - client_df["ds"].min()).total_seconds() / (24 * 3600) if len(client_df) else 0

    base_result = {
        "client": client,
        "status": "ok",
        "n_obs": len(client_df),
        "history_days": history_days,
        "train_rows": 0,
        "test_rows": 0,
        "fit_seconds": np.nan,
        "mae": np.nan,
        "rmse": np.nan,
        "mape_pct": np.nan,
        "wape_pct": np.nan,
        "error": "",
    }

    if history_days < MIN_HISTORY_DAYS or len(client_df) < 24 * MIN_HISTORY_DAYS:
        base_result["status"] = "skipped"
        base_result["error"] = f"Less than {MIN_HISTORY_DAYS} days of active hourly history"
        return base_result, pd.DataFrame(), pd.DataFrame()

    cutoff = client_df["ds"].max() - pd.Timedelta(days=TEST_DAYS)
    train_df = client_df[client_df["ds"] <= cutoff].copy()
    test_df = client_df[client_df["ds"] > cutoff].copy()

    if len(train_df) < 24 * (MIN_HISTORY_DAYS - TEST_DAYS) or len(test_df) < 24:
        base_result["status"] = "skipped"
        base_result["error"] = "Not enough train/test observations after holdout split"
        return base_result, pd.DataFrame(), pd.DataFrame()

    start_time = time.time()
    model = build_prophet_model()
    model.fit(train_df)
    fit_seconds = time.time() - start_time

    holdout_forecast = model.predict(test_df[["ds"]])
    holdout_df = test_df.merge(
        holdout_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]],
        on="ds",
        how="left",
    )
    holdout_df[["yhat", "yhat_lower", "yhat_upper"]] = holdout_df[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)
    holdout_df.insert(0, "client", client)

    metrics = evaluate_predictions(holdout_df["y"], holdout_df["yhat"])

    final_model = build_prophet_model()
    final_model.fit(client_df)
    future = final_model.make_future_dataframe(periods=FUTURE_DAYS * 24, freq=FORECAST_FREQ, include_history=False)
    future_forecast = final_model.predict(future)
    forecast_cols = [
        "ds", "yhat", "yhat_lower", "yhat_upper", "trend",
        "six_hour", "eight_hour", "semi_daily", "daily", "weekly", "monthly", "yearly",
    ]
    future_df = future_forecast[forecast_cols].copy()
    future_df[["yhat", "yhat_lower", "yhat_upper"]] = future_df[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)
    future_df.insert(0, "client", client)

    base_result.update(metrics)
    base_result.update(
        {
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "fit_seconds": fit_seconds,
        }
    )
    return base_result, holdout_df, future_df

## 6. Train Prophet for All Client Columns

Fit models for all 370 clients. Prophet is fitted independently for each client and then refitted on the full history for the future forecast.


In [ ]:
results = []
holdout_frames = []
future_frames = []

run_columns = selected_columns
print(f"Starting Prophet run for {len(run_columns)} client columns...")
print(f"Outputs will be saved in: {OUTPUT_DIR.resolve()}")

for i, client in enumerate(tqdm(run_columns, desc="Prophet models"), start=1):
    try:
        result, holdout_df, future_df = fit_evaluate_forecast_client(client, processed_hourly)
    except Exception as exc:
        result = {
            "client": client,
            "status": "failed",
            "n_obs": np.nan,
            "history_days": np.nan,
            "train_rows": 0,
            "test_rows": 0,
            "fit_seconds": np.nan,
            "mae": np.nan,
            "rmse": np.nan,
            "mape_pct": np.nan,
            "wape_pct": np.nan,
            "error": repr(exc),
        }
        holdout_df = pd.DataFrame()
        future_df = pd.DataFrame()

    results.append(result)
    if not holdout_df.empty:
        holdout_frames.append(holdout_df)
    if not future_df.empty:
        future_frames.append(future_df)

    # Write incremental metrics so a long run still leaves useful partial results.
    pd.DataFrame(results).to_csv(OUTPUT_DIR / "prophet_all_metrics_partial.csv", index=False)

metrics_df = pd.DataFrame(results).sort_values(["status", "wape_pct"], na_position="last").reset_index(drop=True)
holdout_predictions_df = pd.concat(holdout_frames, ignore_index=True) if holdout_frames else pd.DataFrame()
future_forecasts_df = pd.concat(future_frames, ignore_index=True) if future_frames else pd.DataFrame()

metrics_df.to_csv(OUTPUT_DIR / "prophet_all_metrics.csv", index=False)
holdout_predictions_df.to_csv(OUTPUT_DIR / "prophet_all_holdout_predictions.csv", index=False)
future_forecasts_df.to_csv(OUTPUT_DIR / "prophet_all_future_forecasts.csv", index=False)

print("Run complete.")
print(metrics_df["status"].value_counts(dropna=False))
metrics_df.head(10)

## 7. Evaluation Summary

In [ ]:
successful_metrics = metrics_df[metrics_df["status"] == "ok"].copy()

summary_metrics = successful_metrics[["mae", "rmse", "mape_pct", "wape_pct", "fit_seconds", "history_days"]].describe().T
summary_metrics

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

if len(successful_metrics) > 0:
    successful_metrics["mape_pct"].hist(bins=30, ax=axes[0], color="steelblue", edgecolor="black")
    axes[0].set_title("MAPE Distribution")
    axes[0].set_xlabel("MAPE (%)")

    successful_metrics["wape_pct"].hist(bins=30, ax=axes[1], color="seagreen", edgecolor="black")
    axes[1].set_title("WAPE Distribution")
    axes[1].set_xlabel("WAPE (%)")

    successful_metrics["rmse"].hist(bins=30, ax=axes[2], color="coral", edgecolor="black")
    axes[2].set_title("RMSE Distribution")
    axes[2].set_xlabel("RMSE")

    axes[3].scatter(successful_metrics["history_days"], successful_metrics["wape_pct"], alpha=0.7)
    axes[3].set_title("History Length vs WAPE")
    axes[3].set_xlabel("Active history days")
    axes[3].set_ylabel("WAPE (%)")
else:
    for ax in axes:
        ax.text(0.5, 0.5, "No successful models", ha="center", va="center")

plt.tight_layout()

In [ ]:
if len(successful_metrics) > 0:
    display(successful_metrics.nsmallest(10, "wape_pct")[["client", "n_obs", "history_days", "mae", "rmse", "mape_pct", "wape_pct"]])
    display(successful_metrics.nlargest(10, "wape_pct")[["client", "n_obs", "history_days", "mae", "rmse", "mape_pct", "wape_pct"]])

## 8. Holdout Forecast Examples

In [ ]:
if len(successful_metrics) > 0 and not holdout_predictions_df.empty:
    example_clients = [c for c in top_clients if c in successful_metrics["client"].values][:4]
    if not example_clients:
        example_clients = successful_metrics.head(4)["client"].tolist()

    fig, axes = plt.subplots(len(example_clients), 1, figsize=(14, 3.2 * len(example_clients)), sharex=False)
    if len(example_clients) == 1:
        axes = [axes]

    for ax, client in zip(axes, example_clients):
        temp = holdout_predictions_df[holdout_predictions_df["client"] == client]
        ax.plot(temp["ds"], temp["y"], label="Actual", linewidth=1.1)
        ax.plot(temp["ds"], temp["yhat"], label="Prophet forecast", linewidth=1.1)
        ax.fill_between(temp["ds"], temp["yhat_lower"], temp["yhat_upper"], alpha=0.2, label="80% interval")
        wape = successful_metrics.loc[successful_metrics["client"] == client, "wape_pct"].iloc[0]
        ax.set_title(f"Holdout forecast — {client} | WAPE={wape:.2f}%")
        ax.set_ylabel("Load")
        ax.legend(loc="upper right")

    plt.tight_layout()

## 9. Future Forecast Examples

In [ ]:
if len(successful_metrics) > 0 and not future_forecasts_df.empty:
    example_clients = [c for c in top_clients if c in successful_metrics["client"].values][:4]
    if not example_clients:
        example_clients = successful_metrics.head(4)["client"].tolist()

    fig, axes = plt.subplots(len(example_clients), 1, figsize=(14, 3.2 * len(example_clients)), sharex=False)
    if len(example_clients) == 1:
        axes = [axes]

    for ax, client in zip(axes, example_clients):
        history_tail = prepare_client_frame(processed_hourly, client)
        history_tail = history_tail[history_tail["ds"] >= history_tail["ds"].max() - pd.Timedelta(days=14)]
        future_temp = future_forecasts_df[future_forecasts_df["client"] == client]

        ax.plot(history_tail["ds"], history_tail["y"], label="Recent actuals", linewidth=1.1)
        ax.plot(future_temp["ds"], future_temp["yhat"], label="Future forecast", linewidth=1.1)
        ax.fill_between(future_temp["ds"], future_temp["yhat_lower"], future_temp["yhat_upper"], alpha=0.2, label="80% interval")
        ax.axvline(history_tail["ds"].max(), color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{FUTURE_DAYS}-day future forecast — {client}")
        ax.set_ylabel("Load")
        ax.legend(loc="upper right")

    plt.tight_layout()

## 10. Save Metrics


In [ ]:
for path in [
    OUTPUT_DIR / "prophet_all_metrics.csv",
    OUTPUT_DIR / "prophet_all_holdout_predictions.csv",
    OUTPUT_DIR / "prophet_all_future_forecasts.csv",
    OUTPUT_DIR / "prophet_all_metrics_partial.csv",
]:
    print(f"{path}: {'exists' if path.exists() else 'not created yet'}")